# Training LLMs to Play Carrom (ICF Rules) with GRPO + Unsloth

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bp-high/carrom_rl_env/blob/main/examples/grpo_carrom_tutorial.ipynb)

This notebook trains a language model to play Carrom using **GRPO** (Group Relative Policy
Optimization) via TRL + Unsloth on the **OpenEnv Carrom** environment with full
International Carrom Federation (ICF) rule compliance.

## What you'll learn
1. Explore the physics-based Carrom environment (Coulomb friction, ICF rules)
2. Benchmark random and heuristic baselines
3. Fine-tune **Gemma-3-4B** with GRPO (4-bit, Unsloth)
4. Run inference via any OpenAI-compatible endpoint (Nebius, HF, vLLM)
5. Scale up on **Modal** for under $25

## Quick-start cost guide
| Setting | GPU | Est. time | Est. cost |
|---------|-----|-----------|----------|
| Colab T4 (free) | 16 GB | ~20 min | $0 |
| Modal A10G — smoke test | 24 GB | ~15 min | ~$0.30 |
| Modal A10G — blog run | 24 GB | ~2.5 hr | ~$2.75 |
| Modal A100 — best results | 40 GB | ~1.5 hr | ~$5.60 |

## 1. Installation

In [ ]:
# Detect Colab vs local and install accordingly
import subprocess, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Unsloth Colab install (handles CUDA version automatically)
    subprocess.run([sys.executable, '-m', 'pip', 'install',
        'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git',
        '--quiet'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install',
        '--no-deps', 'trl>=0.15', 'peft', 'accelerate', 'bitsandbytes',
        '--quiet'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install',
        'openenv-core', 'pymunk>=6.5', 'numpy', 'pydantic>=2.0',
        '--quiet'], check=True)
    # Install carrom env from the repo root
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '..'], check=True)
else:
    # Local: assume the repo is already installed
    print('Running locally — skipping pip installs.')

print('Setup complete!')

In [ ]:
import sys, os
# Add repo root to path when running from examples/
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import math, random, json, re
import torch

from carrom_env.env import CarromEnv
from carrom_env.models import Action, Observation
from carrom_env.green_agent import GreenCarromAgent
from examples.grpo_utils import (
    format_chat_prompt, parse_response, carrom_reward_for_trl,
    CARROM_SYSTEM_PROMPT,
)

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"})')

## 2. Explore the Environment

Carrom is a physics-based board game with **Coulomb board friction** (constant
deceleration, not viscous drag) and full ICF rule compliance:
- **You play WHITE** coins (+1 pt each)
- **Due rule**: pocketing a black coin returns it to the board, no score, turn ends
- **Queen cover**: pocket queen → must cover with own coin next shot

In [ ]:
env = CarromEnv(seed=42)
obs = env.reset()
print(obs.text_summary)

In [ ]:
# Take a shot and inspect the result
action = Action(placement_x=0.0, angle=0.0, force=0.7)
obs, reward, terminated, truncated, info = env.step(action)

print(f'Reward      : {reward:.3f}')
print(f'Own coins potted : {int(info["coin_potted"])}')
print(f'Due coins (opponent pocketed) : {int(info["due_coins"])}')
print(f'Foul        : {bool(info["foul"])}')
print(f'Sim steps   : {int(info["sim_steps"])}')
print()
print(obs.text_summary)

## 3. Baseline Benchmarks

Establish baselines with random and ICF-aware heuristic policies.

In [ ]:
def random_policy(obs: Observation) -> Action:
    return Action(
        placement_x=random.uniform(-0.35, 0.35),
        angle=random.uniform(-1.0, 1.0),
        force=random.uniform(0.2, 1.0),
    )

def heuristic_policy(obs: Observation) -> Action:
    """ICF-aware heuristic: aim at the nearest WHITE coin to a pocket."""
    best_angle, best_x, best_score = 0.0, 0.0, float('inf')
    baseline_y = -0.5 + 0.08
    for coin in obs.coins:
        if coin.pocketed or coin.color == 'black':  # skip black (due rule)
            continue
        dx, dy = coin.x, coin.y - baseline_y
        score = coin.pocket_distance * (0.5 if coin.color == 'queen' else 1.0)
        if score < best_score:
            best_score = score
            best_angle = math.atan2(dx, dy)
            best_x     = max(-0.35, min(0.35, coin.x * 0.5))
    return Action(placement_x=best_x, angle=best_angle, force=0.6)

print(f'{'Policy':<20} {'Avg Reward':>12} {'Avg Coins':>12} {'Avg Fouls':>10} {'Efficiency':>12}')
print('-' * 70)
for name, policy in [('Random', random_policy), ('Heuristic (ICF)', heuristic_policy)]:
    all_m = []
    for ep in range(5):
        agent = GreenCarromAgent(policy=policy, seed=ep * 42)
        all_m.append(agent.run_episode(max_steps=30, seed=ep * 42))
    avg_r = sum(sum(m.rewards) for m in all_m) / 5
    avg_c = sum(m.coins_potted for m in all_m) / 5
    avg_f = sum(m.total_fouls  for m in all_m) / 5
    avg_e = sum(m.efficiency_score for m in all_m) / 5
    print(f'{name:<20} {avg_r:>12.3f} {avg_c:>12.1f} {avg_f:>10.1f} {avg_e:>12.4f}')

## 4. Generate Training Data

We create diverse board states by stepping random actions from various seeds.

In [ ]:
from datasets import Dataset

def make_dataset(n: int = 300, max_random_steps: int = 6) -> Dataset:
    samples = []
    for seed in range(n * 3):
        if len(samples) >= n:
            break
        env = CarromEnv(seed=seed)
        obs = env.reset()
        for _ in range(random.randint(0, max_random_steps)):
            a = Action(
                placement_x=random.uniform(-0.3, 0.3),
                angle=random.uniform(-0.9, 0.9),
                force=random.uniform(0.25, 0.85),
            )
            obs, _, done, trunc, _ = env.step(a)
            if done or trunc:
                break
        if obs.remaining_coins > 2:
            samples.append({'prompt': format_chat_prompt(obs)})
    return Dataset.from_list(samples)

train_ds = make_dataset(300)
print(f'Training samples : {len(train_ds)}')
print(f'Example prompt (truncated):')
print(train_ds[0]['prompt'][1]['content'][:400])

## 5. Load Gemma-3-4B with Unsloth (4-bit)

Unsloth's optimised 4-bit loading fits **Gemma-3-4B** in ~10 GB VRAM —
well within a Colab T4 (16 GB) or Modal A10G (24 GB).

**Alternative models** (change `MODEL_ID` below):
| Model | VRAM | Quality |
|-------|------|---------|
| `unsloth/gemma-3-1b-it` | 6 GB | Fast demo |
| `unsloth/gemma-3-4b-it` | 10 GB | **Recommended** |
| `unsloth/Qwen2.5-1.5B-Instruct` | 5 GB | Very fast, good JSON |
| `unsloth/Qwen2.5-3B-Instruct` | 8 GB | Good balance |
| `unsloth/Qwen2.5-7B-Instruct` | 18 GB | Best (needs A10G) |

In [ ]:
from unsloth import FastLanguageModel

# Change this to try different models (see table above)
MODEL_ID = 'unsloth/gemma-3-4b-it'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=8,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',  # saves ~30% VRAM vs standard checkpointing
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params : {trainable:,} ({100*trainable/total:.2f}% of {total:,})')

## 6. Reward Function

GRPO needs a reward for each completion. Ours combines:
1. **Format** — valid JSON (+0.3, or -0.5 if unparseable)
2. **Range** — reasonable placement / force / angle (+0.1 each)
3. **Game** — actual environment reward from executing the action
4. **ICF-aware** — extra +0.5 for scoring own coins; -0.3 for each due violation

In [ ]:
# Test the reward function on a few example completions
test_cases = [
    '{"placement_x": 0.0, "angle": 0.15, "force": 0.65}',      # good white-targeting shot
    'I think we should shoot left',                               # unparseable
    '{"placement_x": 0.2, "angle": -0.3, "force": 0.85}',       # valid, aggressive
    '```json\n{"placement_x": -0.1, "angle": 0.05, "force": 0.5}\n```',  # markdown-fenced
]
rewards = carrom_reward_for_trl(test_cases)
for case, r in zip(test_cases, rewards):
    print(f'  {case[:55]:55s}  →  reward = {r:+.3f}')

## 7. GRPO Training

We use TRL's `GRPOTrainer` — no value network needed, just the reward function.

> **Colab T4 users**: `max_steps=100` runs in ~10 min and gives a feel for the training loop.
> For a real result, use `max_steps=500+` or run [the Modal script](#8-scale-up-with-modal).

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir='./carrom-grpo-output',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_generations=4,           # G in GRPO — completions sampled per prompt
    max_prompt_length=512,
    max_completion_length=256,
    max_steps=100,               # increase to 500+ for real training
    learning_rate=5e-6,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    bf16=torch.cuda.is_available(),
    logging_steps=5,
    save_steps=50,
    report_to='none',
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=carrom_reward_for_trl,
    args=training_args,
    train_dataset=train_ds,
)

print('Trainer ready. Starting training...')
trainer.train()
print('Training complete!')

In [ ]:
# Save locally (optional: push to HF Hub)
trainer.save_model('./carrom-grpo-output/final')
tokenizer.save_pretrained('./carrom-grpo-output/final')
print('Saved to ./carrom-grpo-output/final')

# Uncomment to push to HF Hub:
# from huggingface_hub import login
# login()
# trainer.model.push_to_hub('your-username/carrom-grpo-gemma')
# tokenizer.push_to_hub('your-username/carrom-grpo-gemma')

## 8. Scale Up with Modal

For longer runs (500–2000 steps) or bigger models (7B), use the Modal training script.
Everything runs on cloud GPUs; pay only for what you use.

```bash
# Install Modal
pip install modal && modal setup

# Create HF token secret in Modal
modal secret create hf-token HF_TOKEN=hf_...

# Quick smoke-test (200 steps, ~$0.30)
modal run examples/train_modal.py --dry-run   # preview cost
modal run examples/train_modal.py             # run it

# Blog-quality training (2000 steps, ~$2.75)
modal run examples/train_modal.py --steps 2000 --push --repo your-user/carrom-grpo-gemma

# Larger model
modal run examples/train_modal.py --model qwen2.5-7b --gpu a100 --steps 2000
```

All options under $25. See `examples/train_modal.py` for the full model menu.

## 9. Evaluate the Trained Model

In [ ]:
from transformers import pipeline

# Enable native (fast) inference with Unsloth
FastLanguageModel.for_inference(model)

def make_trained_policy(model, tokenizer):
    pipe = pipeline(
        'text-generation',
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=128,
        temperature=0.3,
        do_sample=True,
    )
    def _policy(obs: Observation) -> Action:
        messages = format_chat_prompt(obs)
        result   = pipe(messages)
        text     = result[0]['generated_text'][-1]['content']
        action   = parse_response(text)
        return action if action else Action(placement_x=0.0, angle=0.0, force=0.5)
    return _policy

trained_policy = make_trained_policy(model, tokenizer)

# Compare all policies
N_EVAL = 5
print(f'{'Policy':<22} {'Avg Reward':>12} {'Avg Coins':>10} {'Avg Fouls':>10} {'Efficiency':>12}')
print('-' * 70)
for name, policy in [
    ('Random',          random_policy),
    ('Heuristic (ICF)', heuristic_policy),
    ('GRPO-Trained',    trained_policy),
]:
    ms = [GreenCarromAgent(policy=policy, seed=ep*7).run_episode(max_steps=30, seed=ep*7)
          for ep in range(N_EVAL)]
    print(f'{name:<22}'
          f' {sum(sum(m.rewards) for m in ms)/N_EVAL:>12.3f}'
          f' {sum(m.coins_potted for m in ms)/N_EVAL:>10.1f}'
          f' {sum(m.total_fouls  for m in ms)/N_EVAL:>10.1f}'
          f' {sum(m.efficiency_score for m in ms)/N_EVAL:>12.4f}')

## 10. Inference via OpenAI-Compatible APIs

Once you've pushed a model (or want to use a hosted one), you can plug any
OpenAI-compatible endpoint into the Carrom environment.

### Supported providers (same code, different env vars)

| Provider | `API_BASE_URL` | Key env var |
|----------|---------------|-------------|
| HuggingFace Inference | `https://api-inference.huggingface.co/v1` | `HF_TOKEN` |
| Nebius | `https://api.studio.nebius.com/v1` | `NEBIUS_API_KEY` |
| OpenAI | `https://api.openai.com/v1` | `OPENAI_API_KEY` |
| Local vLLM | `http://localhost:8000/v1` | — |
| Local Ollama | `http://localhost:11434/v1` | — |

In [ ]:
import os, requests

# ── Configure your endpoint here ──────────────────────────────────────────
API_BASE_URL = os.environ.get('API_BASE_URL', 'https://api-inference.huggingface.co/v1')
MODEL_NAME   = os.environ.get('MODEL_NAME',   'Qwen/Qwen3-4B')
API_KEY      = (
    os.environ.get('NEBIUS_API_KEY')
    or os.environ.get('OPENAI_API_KEY')
    or os.environ.get('HF_TOKEN', '')
)
# ──────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """You are a Carrom expert (ICF rules). You play WHITE coins.
Pocket WHITE coins (+1 pt). Pocketing BLACK coins is a DUE — coin returns to board.
Respond with ONLY JSON: {"placement_x": <-0.4..0.4>, "angle": <radians>, "force": <0..1>}"""

def api_call(obs_text: str) -> dict | None:
    try:
        resp = requests.post(
            f'{API_BASE_URL}/chat/completions',
            headers={'Authorization': f'Bearer {API_KEY}', 'Content-Type': 'application/json'},
            json={
                'model': MODEL_NAME,
                'messages': [
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': obs_text},
                ],
                'max_tokens': 128,
                'temperature': 0.3,
            },
            timeout=30,
        )
        resp.raise_for_status()
        text = resp.json()['choices'][0]['message']['content']
        return parse_response(text)
    except Exception as e:
        print(f'API error: {e}')
        return None

def api_policy(obs: Observation) -> Action:
    result = api_call(obs.text_summary)
    if result:
        return Action(**{k: v for k, v in result.__dict__.items()
                        if k in ('placement_x', 'angle', 'force')})
    return Action(placement_x=0.0, angle=0.0, force=0.5)  # fallback

if API_KEY:
    print(f'Running 3 episodes with {MODEL_NAME} ...')
    ms = [GreenCarromAgent(policy=api_policy, seed=ep*5).run_episode(max_steps=15, seed=ep*5)
          for ep in range(3)]
    avg_r = sum(sum(m.rewards) for m in ms) / 3
    avg_c = sum(m.coins_potted for m in ms) / 3
    print(f'Avg reward: {avg_r:.3f}  |  Avg coins: {avg_c:.1f}')
else:
    print('No API key set. Set NEBIUS_API_KEY / OPENAI_API_KEY / HF_TOKEN to test inference.')

## Key Takeaways

1. **Carrom has real ICF physics** — Coulomb friction (constant decel ≠ viscous drag),
   due rule, queen cover, foul handling
2. **Text actions bridge LLMs to physics** — model outputs JSON, env executes physics
3. **GRPO + Unsloth is efficient** — 4-bit + LoRA trains 4B models on a T4 in <20 min
4. **Due rule shapes learning** — model learns to avoid black coins; `-0.3` per due
   in the reward is a surprisingly strong signal
5. **Modal scales cheaply** — 2000-step production run under $3

### Next steps
- Use **torchforge** for distributed multi-GPU rollout collection
- Try **vision input** (rendered board state) for multimodal training
- Tune `num_generations` (G) — higher G = better gradient signal but more memory
- Push environment to the OpenEnv Hub: `openenv push --repo-id openenv/carrom`